# Baseline Auto-tuning Demo (v0.6.2+)

This notebook demonstrates the `autotune_baseline` / `apply_choice` workflow introduced in v0.6.2.
It covers both the **mapping** (`RamanMapping`) and **single-fit** (`RamanFit`) use cases using
fully synthetic data — no WDF files required.

### Workflow summary

1. Create a fit object with a placeholder baseline.
2. Call `autotune_baseline()` to score a grid of baseline candidates on a representative spectrum.
3. Inspect the RMSE ranking and select a winner.
4. Call `apply_choice(result.winner)` to commit the best baseline.
5. Run `fit_spectra()` / `fit_spectrum()` as normal.

**Prerequisites:** `pip install -e .` (base install — no extras required).

In [ ]:
import sys
from pathlib import Path

# Ensure the worktree's src/ takes priority over the editable install so new
# v0.6.2 features are available when running from the development worktree.
_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import matplotlib
matplotlib.use('Agg')  # headless backend for CI

import numpy as np
import matplotlib.pyplot as plt

from ramanpl.mapping import RamanMapping
from ramanpl.single_fit.RamanFit import RamanFit

print(f"RamanPL_2D version: {__import__('ramanpl').__version__}")

## 1. Synthetic data

A single Lorentzian peak at 400 cm⁻¹ on a sloped baseline, replicated into a 3×3 mapping cube.

In [ ]:
rng = np.random.default_rng(42)

x = np.linspace(300.0, 500.0, 120)

def _lorentzian(x, center, fwhm, amplitude):
    gamma = fwhm / 2.0
    return amplitude * gamma**2 / ((x - center)**2 + gamma**2)

# Sloped background + Lorentzian peak + noise
background = 5.0 + 0.03 * (x - 300.0)
peak = _lorentzian(x, 400.0, 12.0, 120.0)
noise = rng.normal(0.0, 1.5, size=x.shape)
y_single = background + peak + noise

# 3x3 cube with small per-pixel variations
rows, cols = 3, 3
cube = np.stack(
    [y_single + rng.normal(0.0, 0.5, size=x.shape) for _ in range(rows * cols)],
    axis=0,
).reshape(rows, cols, len(x))

peak_bounds = {"A": ([388.0, 3.0, 0.01], [415.0, 25.0, 300.0])}

print(f"Spectral axis: {x[0]:.1f} – {x[-1]:.1f} cm⁻¹  ({len(x)} points)")
print(f"Cube shape: {cube.shape}")

## 2. RamanMapping — baseline auto-tuning

### 2a. Create the mapping object with a placeholder baseline

In [ ]:
mapping = RamanMapping.from_arrays(
    cube, x, cols, rows,
    custom_peaks=peak_bounds,
    data_range=(300.0, 500.0),
    background_remove=True,
    baseline_method={"method": "poly", "poly_order": 1},
    smoothing=False,
)
print("Mapping created. Baseline method:", mapping.baseline_method)

### 2b. Score the baseline grid on seed pixel (1, 1)

`autotune_baseline` is **read-only** — it never modifies the object.

In [ ]:
result = mapping.autotune_baseline(
    seed_coord=(1, 1),
    methods=["poly", "airpls", "gaussian"],  # restrict for speed; omit for full 24-candidate grid
    plot=False,
)

print(f"Candidates scored: {result.meta['n_candidates']}")
print(f"Seed pixel: {result.seed_coord}")
print()
print("Top-5 ranking:")
for rank, entry in enumerate(result.ranking[:5], start=1):
    print(f"  #{rank:2d}  method={entry['method']:8s}  kwargs={entry['kwargs']}  RMSE={entry['rmse']:.4f}")

print()
print("Winner:", result.winner)

### 2c. Verify object is unchanged after `autotune_baseline`

In [ ]:
print("Baseline method after autotune (unchanged):", mapping.baseline_method)
assert mapping.baseline_method == {"method": "poly", "poly_order": 1}, "Object was mutated!"
print("OK — object not mutated by autotune_baseline")

### 2d. Commit the winner and run the full map fit

In [ ]:
mapping.apply_choice(result.winner)
print("Baseline after apply_choice:", mapping.baseline_method)

mapping.fit_spectra(
    warm_start=True,
    fit_spectrum_kwargs=dict(
        adaptive_multistart=False,
        n_starts=1,
        diagnostics="light",
    ),
)
print("Fit complete.")

### 2e. Fit summary and feature table

In [ ]:
rep = mapping.fit_summary()
print(rep)

df = mapping.feature_table()
print()
print("Feature table shape:", df.shape)
df.head()

### 2f. Export — verify provenance block

After `apply_choice`, the TXT export includes a `baseline_autotune:` block.

In [ ]:
import tempfile, os

with tempfile.NamedTemporaryFile(suffix=".txt", delete=False) as f:
    export_path = f.name

mapping.export_fit_map(export_path)

with open(export_path) as f:
    header_lines = [line for line in f if line.startswith("#")]

os.unlink(export_path)

autotune_lines = [l for l in header_lines if "baseline_autotune" in l or "winner" in l]
print("Provenance lines containing autotune info:")
for l in autotune_lines:
    print(" ", l.rstrip())

assert any("baseline_autotune" in l for l in header_lines), "baseline_autotune block missing from export!"
print()
print("OK — baseline_autotune provenance block present in export.")

## 3. RamanFit — baseline auto-tuning

The single-fit workflow is identical but operates on one spectrum.

### 3a. Create a RamanFit object

In [ ]:
fit = RamanFit(
    spectra=y_single,
    wavenumber=x,
    custom_peaks=peak_bounds,
    background_remove=True,
    baseline_method={"method": "poly", "poly_order": 1},
    smoothing=False,
)
print("RamanFit created. Baseline method:", fit.baseline_method)

### 3b. Score the baseline grid

In [ ]:
result_fit = fit.autotune_baseline(
    methods=["poly", "airpls", "gaussian"],
    plot=False,
)

print(f"Candidates scored: {result_fit.meta['n_candidates']}")
print()
print("Top-5 ranking:")
for rank, entry in enumerate(result_fit.ranking[:5], start=1):
    print(f"  #{rank:2d}  method={entry['method']:8s}  kwargs={entry['kwargs']}  RMSE={entry['rmse']:.4f}")

print()
print("Winner:", result_fit.winner)

### 3c. Commit and fit

In [ ]:
fit.apply_choice(result_fit.winner)
print("Baseline after apply_choice:", fit.baseline_method)

fit.fit_spectrum(n_starts=2)
diag = getattr(fit, "fit_diagnostics", {}) or {}
print("Fit complete. RMSE:", round(float(diag.get("rmse", float("nan"))), 4))

### 3d. Verify `apply_choice` refreshed derived attributes

In [ ]:
assert fit.processed_spectra is not None, "processed_spectra not set"
assert fit.peak_intensity > 0, "peak_intensity not refreshed"
assert fit.intensity_normal is not None, "intensity_normal not set"
print("Peak intensity:", round(float(fit.peak_intensity), 3))
print("Wavenumber range: {:.1f} – {:.1f} cm⁻¹".format(fit.wavenumber[0], fit.wavenumber[-1]))
print("OK — derived attributes refreshed by apply_choice.")

## 4. Summary

Both `RamanMapping` and `RamanFit` support the same two-step baseline auto-tuning pattern:

| Step | Call | Effect |
|------|------|--------|
| 1 | `autotune_baseline(...)` | Scores grid, returns ranked result — **object unchanged** |
| 2 | `apply_choice(result.winner)` | Commits winner, invalidates cache, re-applies pipeline |

After `apply_choice`, `fit_spectra()` / `fit_spectrum()` run with the updated baseline.
The export TXT header includes a `baseline_autotune:` provenance block documenting the winner.